# CE541E08 — Unit 3 · Day 25 — Reshape, Transpose, vstack and Standardised Anomalies

| | |
|---|---|
| **Course** | CE541E08 |
| **Department** | Civil Engineering · Christ University |
| **Instructor** | Dr. Arpan Pradhan |
| **Unit** | Unit 3 — The NumPy Library |
| **Session** | Day 25 of 45 |
| **CO** | CO3, CO4 |
| **Topics** | reshape · transpose · vstack / concatenate · standardised anomalies |

---
> Read the explanation before each code block. Check the expected output. Run the cell and verify. Then try the small challenge at the end.
---

In [ ]:
student_name = "Your Full Name"
roll_number  = "2024XXXXXX"
session      = "Day 25"
print(f"CE541E08 | {student_name} | {roll_number} | {session}")

---
## Section 1 — Restructuring Arrays

Today we learn how to **change the shape** of an array without changing its data:

| Operation | What it does | When to use it |
|---|---|---|
| `reshape(r, c)` | Rearranges elements into r rows × c columns | Convert a flat daily series into a monthly matrix |
| `.T` (transpose) | Swaps rows and columns | Change from years×months to months×years |
| `np.vstack([a,b])` | Stacks arrays vertically (adds rows) | Combine river basin arrays into one matrix |
| `np.concatenate([a,b])` | Joins arrays end-to-end | Combine July and August into one series |

None of these operations copy the data — they just change how NumPy interprets the memory layout. This makes them very fast even for large datasets.

---
## Code Block 1 — reshape: Daily Series to Monthly Matrix

### What this code does

We generate a synthetic 360-day rainfall series (12 months × 30 days) and reshape it into a `(12,30)` matrix. Then we use `axis` operations to compute monthly totals and monthly peak values — exactly as you would process a year of daily gauge data.

### Why each step is taken

**`np.random.seed(42)`:**
Sets the random number generator to a fixed starting point. This ensures the same random values are generated every time the cell is run — essential for reproducible results in teaching and testing.

**`np.concatenate([...])`:**
Joins 12 separate monthly arrays (each 30 values long) into one flat array of 360 values. Each monthly sub-array uses `np.random.exponential(scale, 30)` with a different scale to simulate the seasonal pattern — low in winter, high in monsoon.

**`reshape(12, 30)`:**
Converts the flat 360-element array into a `(12,30)` matrix — 12 rows (months) × 30 columns (days). The data is read row by row: days 1–30 become row 0 (January), days 31–60 become row 1 (February), and so on.

**`matrix.sum(axis=1)` and `matrix.max(axis=1)`:**
After reshaping, `axis=1` operates across the 30 daily columns — giving one value per month row. `sum(axis=1)` gives the monthly total; `max(axis=1)` gives the monthly peak.

### Algorithm

```
1. np.random.seed(42) — fix randomness for reproducibility

2. Generate 12 monthly sub-arrays of 30 values each
   scale varies by month to simulate seasonal pattern
   np.concatenate joins them into a flat (360,) array

3. reshape(12, 30)
   → (360,) becomes (12,30): months × days

4. matrix.sum(axis=1)  → monthly totals, shape (12,)
   matrix.max(axis=1)  → monthly peak,   shape (12,)

5. Print shape at each step and the computed statistics
```

### Expected output

```
Daily shape : (360,)
Matrix shape: (12, 30)
Monthly totals: [  97.   165.   318.  1039.  2022.  3371.  3117.  3059.  2233.  1718.   704.   178.]
Monthly peaks : [  20.7   36.7   53.1  195.3  380.7  546.3  534.8  500.8  426.9  295.3  139.0   41.5]
```

In [ ]:
import numpy as np

np.random.seed(42)

# Generate 360 days of synthetic rainfall (mm)
# Each month gets 30 values from an exponential distribution
# Scale parameter controls the seasonal intensity
daily_360 = np.concatenate([
    np.random.exponential(s, 30)
    for s in [5, 8, 15, 45, 80, 120, 115, 110, 85, 65, 30, 8]
])
daily_360 = np.round(daily_360, 1)

months = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
print(f"Daily shape : {daily_360.shape}")

# reshape(12, 30): reorganise 360 values into 12 rows × 30 columns
# Row 0 = January (days 1-30), Row 1 = February (days 31-60), etc.
matrix = daily_360.reshape(12, 30)
print(f"Matrix shape: {matrix.shape}")

# axis=1: operate across the 30 daily columns → one value per month row
print("Monthly totals:", matrix.sum(axis=1).round(0))
print("Monthly peaks :", matrix.max(axis=1).round(1))

### 🔁 Try this

Compute the number of **rainy days per month** (days where rainfall > 0).

Use `(matrix > 0).sum(axis=1)` — this gives a boolean matrix then counts True values across each row.

Which month has the most rainy days?

---
## Code Block 2 — Transpose: Switching from Years×Months to Months×Years

### What this code does

We transpose a `(3,12)` rainfall matrix — originally years × months — to a `(12,3)` matrix — months × years. This lets us iterate over months instead of years and compute inter-annual statistics for each month.

### Why each step is taken

**`.T` — transpose:**
Swaps rows and columns. A `(3,12)` matrix becomes `(12,3)`. No data is copied — only the way NumPy interprets the array shape changes. This is a very cheap operation.

**Why transpose for monthly statistics:**
When the original matrix has years as rows and months as columns, iterating over rows gives one year at a time. After transposing, iterating over rows gives one month at a time — which is what we want when computing inter-annual statistics (mean and variability) for each calendar month.

**`.mean()` and `.std()` on each row after transpose:**
Each row of `data_T` is the 3-year record for one month. `.mean()` gives the 3-year climatological mean for that month. `.std()` gives the inter-annual variability.

### Algorithm

```
1. Create (3,12) data matrix — years × months

2. data_T = data.T
   → shape becomes (12,3) — months × years

3. Iterate over rows of data_T:
   each row = [year1_value, year2_value, year3_value] for one month
   row.mean() → 3-year mean for that month
   row.std()  → inter-annual variability for that month

4. Print month name, values, mean, std
```

### Expected output

```
Original   : (3, 12)  (years x months)
Transposed : (12, 3)  (months x years)
  Jan: [8 5 10]  mean=7.7  std=2.1
  Feb: [12  8 15]  mean=11.7  std=2.9
  Mar: [18 22 16]  mean=18.7  std=2.5
  ...
  Dec: [13 10 16]  mean=13.0  std=2.4
```

In [ ]:
import numpy as np

# 3-year × 12-month rainfall matrix
data = np.array([
    [8, 12,18,52,87,134,118,113,95,71,44,13],   # Year 1
    [5,  8,22,48,92,145,123,108,89,68,38,10],   # Year 2
    [10,15,16,55,83,128,115,119,98,74,48,16],   # Year 3
])

print(f"Original   : {data.shape}  (years x months)")

# .T swaps rows and columns — (3,12) becomes (12,3)
# No data is copied; only the shape interpretation changes
data_T = data.T
print(f"Transposed : {data_T.shape}  (months x years)")

months = ['Jan','Feb','Mar','Apr','May','Jun',
          'Jul','Aug','Sep','Oct','Nov','Dec']

# After transpose, each row = one month across all 3 years
for i, m in enumerate(months):
    row = data_T[i]   # 3 values: Year1, Year2, Year3 for month i
    print(f"  {m}: {row}  mean={row.mean():.1f}  std={row.std():.1f}")

### 🔁 Try this

Which month has the **highest inter-annual variability** (highest std)?

Use: `std_per_month = data_T.std(axis=1)` then `months[std_per_month.argmax()]`

Does this match your expectation given the seasonal rainfall pattern?

---
## Code Block 3 — vstack and concatenate: Combining Datasets

### What this code does

We combine monthly rainfall series from three river basins into a single matrix using `np.vstack`, and we concatenate July and August daily streamflow into one combined series using `np.concatenate`. These are the two most common ways to join arrays in hydrology data processing.

### Why each step is taken

**`np.vstack([cauvery, tungabhadra, krishna])`:**
Stacks three 1-D arrays of length 12 vertically — producing a `(3,12)` matrix. Each input becomes one row. `vstack` is equivalent to `np.row_stack` and is used when combining arrays of the same column count.

**`basin_matrix.sum(axis=1)` and `basin_matrix[:,5:9].sum(axis=1)`:**
After stacking, all 3 basins can be analysed together with a single `axis=1` operation. The monsoon slice `[:,5:9]` selects June–September columns for all 3 basins simultaneously.

**`np.concatenate([jul, aug])`:**
Joins two 1-D arrays end-to-end into a longer 1-D array. July (10 values) + August (10 values) → combined (20 values). Unlike `vstack`, `concatenate` keeps the data 1-D.

### Algorithm

```
1. Define 3 basin arrays — each shape (12,)

2. np.vstack([a, b, c])
   → shape (3,12) — one row per basin

3. basin_matrix.sum(axis=1)
   → annual total per basin, shape (3,)

4. basin_matrix[:,5:9].sum(axis=1) / basin_matrix.sum(axis=1) * 100
   → monsoon percentage per basin, shape (3,)

5. np.concatenate([jul, aug])
   → flat (20,) array joining both months
```

### Expected output

```
Basin matrix shape: (3, 12)
  Cauvery        : 765 mm/yr  monsoon=59.1%
  Tungabhadra    : 876 mm/yr  monsoon=62.8%
  Krishna        : 649 mm/yr  monsoon=62.9%
Jul+Aug combined: (20,)
```

In [ ]:
import numpy as np

# Monthly rainfall (mm) for 3 river basins — each is a 1-D array of 12 months
cauvery     = np.array([8, 12,18,52,87,134,118,113,95,71,44,13])
tungabhadra = np.array([12,16,22,58,95,142,126,120,102,78,50,15])
krishna     = np.array([6,  9,14,38,72,118,105, 99, 82,60,36,10])

# vstack stacks 1-D arrays as rows → (3,12) matrix
basin_matrix = np.vstack([cauvery, tungabhadra, krishna])
names = ['Cauvery', 'Tungabhadra', 'Krishna']
print(f"Basin matrix shape: {basin_matrix.shape}")

for n, row in zip(names, basin_matrix):
    annual  = row.sum()
    monsoon = row[5:9].sum()
    pct     = monsoon / annual * 100
    print(f"  {n:<15}: {annual} mm/yr  monsoon={pct:.1f}%")

# concatenate joins 1-D arrays end-to-end → longer 1-D array
jul = np.array([234, 267, 312, 890, 1245, 987, 756, 543, 412, 345])
aug = np.array([289, 245, 212, 198, 220, 265, 310, 456, 678, 890])
combined = np.concatenate([jul, aug])
print(f"Jul+Aug combined: {combined.shape}")

### 🔁 Try this

Add a fourth basin — Krishna (already defined above, but with different values for variety):

```python
godavari = np.array([10,14,20,55,90,148,132,125,108,82,52,16])
```

- Use `np.vstack` to add it as a 4th row
- Recompute annual totals and monsoon percentages for all 4 basins

---
## Code Block 4 — Standardised Anomalies

### What this code does

We generate 10 years of synthetic monthly rainfall, compute the climatological mean and standard deviation for each month, then compute the standardised anomaly (z-score) for every year-month combination. A z-score above +1 indicates a significantly wet month; below -1 indicates a significantly dry month.

### Why each step is taken

**`np.vstack([...])` with a list comprehension:**
Generates 10 rows (one per year), each computed by adding small random perturbations to the baseline. `np.maximum(..., 0)` prevents negative rainfall values.

**`clim = data_10yr.mean(axis=0)`:**
Averaging across the 10 years (axis=0) gives the 10-year climatological mean for each of the 12 months — shape `(12,)`.

**`std = data_10yr.std(axis=0)`:**
Standard deviation across years — measures how much each month's rainfall typically varies year to year.

**`anomaly = (data_10yr - clim) / std`:**
Broadcasting: `clim` and `std` are both shape `(12,)`, while `data_10yr` is `(10,12)`. NumPy broadcasts them across all 10 rows. The result is the z-score for every year-month cell.

### Algorithm

```
1. Generate (10,12) synthetic rainfall matrix
   base = long-term monthly means
   Each row = base + small random perturbation

2. clim = data_10yr.mean(axis=0)  → shape (12,) 10-year mean per month
   std  = data_10yr.std(axis=0)   → shape (12,) inter-annual std per month

3. anomaly = (data_10yr - clim) / std
   Broadcasting: (10,12) - (12,) / (12,) → (10,12)
   = z-score for every year × month

4. Print table: years as rows, months as columns
   Flag wet (>+1) and dry (<-1) cells
```

### Expected output

```
Standardised anomalies (>1=wet, <-1=dry):
  Year      J      F      M      A      M      J      J      A      S      O      N      D
  2015  +0.3  +1.2  -0.8  +0.5  -1.1  +0.2  -0.4  +0.9  +0.1  -0.7  +1.3  -0.2
  ...
```

In [ ]:
import numpy as np

np.random.seed(0)
years = list(range(2015, 2025))
# Long-term monthly means (mm)
base = np.array([8, 12, 18, 52, 87, 134, 118, 113, 95, 71, 44, 13])

# Generate 10 years: each row = base + small random variation (±15%)
# np.maximum ensures no negative rainfall values
data_10yr = np.vstack([
    base + np.random.normal(0, base * 0.15)
    for _ in years
])
data_10yr = np.round(np.maximum(data_10yr, 0), 1)

# Climatology: mean and std across 10 years for each month
# axis=0: average across rows (years), one value per column (month)
clim = data_10yr.mean(axis=0)   # shape (12,) — 10-year mean per month
std  = data_10yr.std(axis=0)    # shape (12,) — inter-annual std per month

# Standardised anomaly (z-score):
# Broadcasting: (10,12) - (12,) → (10,12) — clim subtracted from each row
# Then divided by std — also broadcast across all rows
anomaly = (data_10yr - clim) / std

months = ['J','F','M','A','M','J','J','A','S','O','N','D']
print("Standardised anomalies (>1=wet, <-1=dry):")
print(f"{'Year':>6} " + " ".join(f"{m:>5}" for m in months))
for yr, row in zip(years, anomaly):
    print(f"{yr:>6} " + " ".join(f"{v:>+5.1f}" for v in row))

### 🔁 Try this

Count how many **significantly wet months** (z > 1.0) each year had:

Use `(anomaly > 1.0).sum(axis=1)` — gives a count per year.

Which year was overall the wettest? Which was the driest?

---
## Session Summary — Reshape, Transpose, Stack

| Operation | Syntax | Input → Output shape |
|---|---|---|
| Reshape to matrix | `arr.reshape(12,30)` | `(360,)` → `(12,30)` |
| Reshape to column | `arr.reshape(-1,1)` | `(7,)` → `(7,1)` |
| Transpose | `data.T` | `(3,12)` → `(12,3)` |
| Stack as rows | `np.vstack([a,b,c])` | 3×`(12,)` → `(3,12)` |
| Join end-to-end | `np.concatenate([a,b])` | `(10,)`,`(10,)` → `(20,)` |
| Mean across years | `data.mean(axis=0)` | `(10,12)` → `(12,)` |
| Std across years | `data.std(axis=0)` | `(10,12)` → `(12,)` |
| Z-score | `(data-clim)/std` | Broadcasting `(10,12)` |
| Rainy days per month | `(matrix>0).sum(axis=1)` | `(12,30)` → `(12,)` |

---
## Day 25 Assignment

July (31 days) and August (31 days) streamflow arrays:

```python
july   = np.array([234,267,312,890,1245,987,756,543,412,345,289,245,212,198,220,
                   265,310,456,678,890,1123,987,765,543,421,345,289,245,212,198,210])
august = np.array([289,312,456,678,890,1123,987,765,543,421,389,345,312,289,265,
                   243,221,289,367,456,567,678,543,421,345,289,245,212,198,210,234])
```

1. Concatenate July and August into one combined 62-day array
2. Reshape into a `(2,31)` matrix — one row per month
3. Compute mean flow for each month using `axis=1`
4. Find which of the 31 calendar day positions had consistently higher flow — use `matrix.mean(axis=0).argmax() + 1`

### ▶ Assignment cell

In [ ]:
import numpy as np

july   = np.array([234,267,312,890,1245,987,756,543,412,345,289,245,212,198,220,
                   265,310,456,678,890,1123,987,765,543,421,345,289,245,212,198,210])
august = np.array([289,312,456,678,890,1123,987,765,543,421,389,345,312,289,265,
                   243,221,289,367,456,567,678,543,421,345,289,245,212,198,210,234])

combined     = ???          # concatenate july and august
matrix       = ???          # reshape to (2,31)
monthly_mean = ???          # mean across days for each month (axis=1)
monthly_max  = ???          # max across days for each month (axis=1)
best_day     = matrix.mean(axis=0).argmax() + 1   # 1-based day number

print(f"Matrix shape  : {matrix.shape}")
print(f"Monthly mean  : {monthly_mean.round(1)}")
print(f"Monthly max   : {monthly_max}")
print(f"Best day      : Day {best_day}")

---
- [ ] Run all cells from top to bottom — verify outputs match expected outputs above
- [ ] Complete the assignment cell (replace `???` placeholders)
- [ ] Upload to GitHub: `Unit3_NumPy/CE541E08_U3_Day25.ipynb`
- [ ] Commit message: `Day 25 assignment completed`

*CE541E08 · Civil Engineering · Christ University · 2026-27 · Dr. Arpan Pradhan*